# Directory Pipeline — Enriched Pipeline Walkthrough

This notebook walks through the three steps that add **spatial coordinates** to your extracted entries — linking each record back to its exact location on the source page image.

| Step | Flag | What it does | Output |
|------|------|-------------|--------|
| 1 | `--surya-ocr` | Neural net draws bounding boxes around every text line | `*_surya.json` |
| 2 | `--align-ocr` | Maps Gemini's transcribed text onto those boxes | `*_aligned.json` |
| 3 | `--review-alignment` | Interactive UI to fix bad matches | updates `*_aligned.json` |

After completing these steps, every entry in your CSV will have a `canvas_fragment` URL containing a precise `#xywh=` bounding box — click it to open the IIIF viewer at that exact spot on the original page.

---

### Before you start

You should already have run the basic pipeline steps for your volume:
- `--download` — page images exist in `output/<slug>/`
- `--gemini-ocr` — `*_gemini.txt` files exist alongside the images
- `--extract-entries` — an `entries_*.csv` exists (we'll re-run this at the end with bboxes)

If you haven't done those yet, see the `README.md` or `CLAUDE.md` in the pipeline root.

---

> ⚠️ **Surya OCR requires a GPU or Apple Silicon.** On a CPU it will run, but very slowly — plan for hours rather than minutes on a large volume. If you're using Google Colab, see `colab/ocr-align-review.ipynb` for the GPU-specific setup. The alignment and review steps (Steps 2–3) run fine on any hardware.

---
## Setup

Run all cells in this section before anything else.

In [ ]:
import os
from pathlib import Path

# ── Edit these values ─────────────────────────────────────────────────────────

# Path to the directory-pipeline folder.
# If you launched Jupyter from inside the pipeline folder, "." works as-is.
PIPELINE_DIR = "."

# The model name used when you ran Gemini OCR.
# Check one of your *_gemini.txt filenames — the model name is embedded in it.
MODEL = "gemini-3.1-flash-lite-preview"

# Port for the review server (Step 3). 5000 is the default.
PORT = 5000

# ─────────────────────────────────────────────────────────────────────────────

PIPELINE_DIR = str(Path(PIPELINE_DIR).resolve())
os.chdir(PIPELINE_DIR)
print("✓ Working directory:", os.getcwd())

In [ ]:
# Install pipeline dependencies.
# surya-ocr pulls in PyTorch — this takes a few minutes the first time.
# If you already have these installed in your environment, this cell is a no-op.
%pip install -q google-genai pillow requests python-dotenv flask geopy
%pip install -q "surya-ocr==0.17.1" "transformers>=4.56.1,<5.0"
print("\n✓ Dependencies installed")

### Choose your volume

Enter the path to your volume **relative to the `output/` folder**.

For NYPL collections this is usually two levels deep:
```
the_green_book_9ea5d5b0/d440c0b0-1a71-0132-7f48-58d385a7bbd0
```
For Internet Archive collections it's typically one level:
```
brewers_guide_for_the_united_states_cana_bd047d10/bd047d10
```

Not sure of the path? Run this to find your Gemini OCR files:
```
!find output -name "*_gemini.txt" | head -10
```

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

VOLUME = None

slug_widget = widgets.Text(
    value="",
    placeholder="e.g. the_green_book_9ea5d5b0/d440c0b0-1a71-0132-7f48-58d385a7bbd0",
    description="Volume path:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="700px"),
)

status_out = widgets.Output()

def _on_change(change):
    global VOLUME
    slug = change["new"].strip().strip("/")
    if not slug:
        return
    VOLUME = f"output/{slug}"
    vol_path = Path(PIPELINE_DIR) / VOLUME
    with status_out:
        status_out.clear_output()
        images  = list(vol_path.glob("*.jpg")) + list(vol_path.glob("*.png"))
        gemini  = list(vol_path.glob("*_gemini.txt"))
        surya   = list(vol_path.glob("*_surya.json"))
        aligned = list(vol_path.glob("*_aligned.json"))
        print(f"Volume: {VOLUME}")
        print(f"  {len(images):4d} page images")
        print(f"  {len(gemini):4d} Gemini OCR files  (*_gemini.txt)")
        print(f"  {len(surya):4d} Surya files        (*_surya.json)  — Step 1 output")
        print(f"  {len(aligned):4d} Aligned files      (*_aligned.json) — Step 2 output")
        if not vol_path.exists():
            print("\n  ⚠ Path not found — check the volume path above")
        elif not images:
            print("\n  ⚠ No images found — check that download completed")
        elif not gemini:
            print("\n  ⚠ No Gemini OCR files — run --gemini-ocr before this step")

slug_widget.observe(_on_change, names="value")
display(slug_widget, status_out)

---
## Step 1: Surya OCR — detect bounding boxes

Gemini reads the text on each page but doesn't know *where* on the page each line sits. Surya fills that gap: it's a neural network that looks at the page image and draws a bounding box around every line of text it finds.

After this step, each page will have a `*_surya.json` file containing a list of `(x1, y1, x2, y2)` boxes — one per detected text line. Those boxes are what eventually get attached to entries in your final CSV.

**Batch size guidance** — higher is faster, but risks out-of-memory errors:

| Hardware | Photographic / book scans | Microfilm / high-contrast |
|----------|--------------------------|---------------------------|
| NVIDIA T4 (Colab free) | 8–16 | 32–64 |
| NVIDIA L4 | 32–72 | 72–128 |
| Apple Silicon (M1/M2/M3) | 16–32 | 32–64 |
| CPU only | 1–4 | 4–8 |

Start conservatively. If you get an out-of-memory error, lower `--batch-size` and re-run — **Surya skips pages that already have a `*_surya.json`**, so no work is lost.

**Expected time:** A few seconds per page on a GPU. A 100-page volume takes roughly 5–15 minutes on a T4.

In [ ]:
# Adjust --batch-size to fit your hardware (see guidance above).
# 16 is a safe starting point for a GPU; drop to 4 for CPU.
!time python main.py --batch-size 4 --surya-ocr {VOLUME}

In [ ]:
# Verify all pages were processed
from pathlib import Path

vol_path = Path(PIPELINE_DIR) / VOLUME
images = sorted(list(vol_path.glob("*.jpg")) + list(vol_path.glob("*.png")))
surya_files = sorted(vol_path.glob("*_surya.json"))

print(f"Page images:  {len(images)}")
print(f"Surya JSON:   {len(surya_files)}")

missing = [
    img.name for img in images
    if not (vol_path / (img.stem + "_surya.json")).exists()
]
if missing:
    print(f"\n⚠ {len(missing)} pages still missing Surya output:")
    for m in missing[:10]:
        print(f"  {m}")
    if len(missing) > 10:
        print(f"  ... and {len(missing) - 10} more")
    print("Re-run the cell above to process them.")
else:
    print("\n✓ All pages have Surya output — ready for alignment")

---
## Step 2: Align OCR — match text to bounding boxes

We now have two things for each page:
- **Gemini's text** — accurate transcription, in reading order, but no position info
- **Surya's boxes** — precise positions on the page, but no text content

Alignment connects them. It uses a sequence-alignment algorithm (similar to how bioinformatics tools align DNA strands) to find the best match between Gemini's lines and Surya's boxes, using city and state headings as anchor points. A second pass then tries to recover any lines that were missed in the first.

The output is a `*_aligned.json` file for each page — every matched line has both text and coordinates. Lines that couldn't be matched appear in an `unmatched_gemini` list and won't get coordinates in the final CSV.

**Confidence levels** you'll see in the output:
- `"word"` — high-confidence automatic match
- `"line"` — moderate-confidence automatic match  
- `"manual"` — you confirmed or corrected this match in the review UI (Step 3)

**Expected time:** Much faster than Surya — typically a few minutes for most volumes, even on CPU.

In [ ]:
!time python main.py --align-ocr --model {MODEL} {VOLUME}

In [ ]:
# Summarize alignment quality across the volume
import json
from pathlib import Path

vol_path = Path(PIPELINE_DIR) / VOLUME
aligned_files = sorted(vol_path.glob("*_aligned.json"))
print(f"Aligned JSON files: {len(aligned_files)}")

if aligned_files:
    total_matched = total_unmatched = flagged = 0
    problem_pages = []

    for af in aligned_files:
        data = json.loads(af.read_text())
        matched   = len(data.get("lines", []))
        unmatched = len(data.get("unmatched_gemini", []))
        flag      = data.get("possible_column_merge", False)
        total_matched   += matched
        total_unmatched += unmatched
        if flag:
            flagged += 1
        # Flag pages where more than half the lines went unmatched (and there's real content)
        if matched > 5 and unmatched > matched * 0.5:
            problem_pages.append((af.name, matched, unmatched))

    total = total_matched + total_unmatched
    pct = total_matched / total * 100 if total else 0
    print(f"\nOverall match rate:             {total_matched}/{total} lines ({pct:.0f}%)")
    print(f"Pages flagged for column merge: {flagged}")

    if problem_pages:
        print(f"\nPages with >50% unmatched lines ({len(problem_pages)} total):")
        for name, m, u in sorted(problem_pages, key=lambda x: -x[2])[:10]:
            print(f"  {name}  matched={m}  unmatched={u}")
        print("\nThese are good candidates for manual review in Step 3.")
    else:
        print("\n✓ No pages with severe alignment problems detected")

---
## Step 3: Review Alignment — fix problem pages

Alignment works automatically on most pages, but some layouts trip it up: two-column text, unusual formatting, faint print, or pages where Surya missed an entire column. The review UI lets you inspect these pages and manually correct bad matches.

**How the UI works:**
1. Pages are sorted by problem severity — flagged and high-unmatched pages come first
2. Each page shows the scan on the left and matched text lines on the right
3. You can click to link a text line to a different bounding box, or mark a line as confirmed
4. Click **Done** on a page to save your corrections and move to the next

The review tool is a Flask web app. It starts a local server and you open it in your browser.

> 💡 **You don't need to review every page.** Focus on pages the quality check above flagged, and any obvious errors you notice. Good-enough alignment on most pages is more useful than perfect alignment on a few.

In [ ]:
import subprocess
import time

# Stop any server already running on this port
subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)
time.sleep(1)

# Start the review server in the background
subprocess.Popen(
    ["python", "-m", "pipeline.review_alignment",
     VOLUME,
     "--host", "0.0.0.0",
     "--port", str(PORT),
     "--model", MODEL],
    cwd=PIPELINE_DIR,
    stdout=open("/tmp/review.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print(f"Server starting on: {VOLUME}")
print(f"Loading Surya models takes ~30 seconds.")
print(f"\nOnce ready, open this URL in your browser:")
print(f"  http://localhost:{PORT}")
print(f"\nRun the next cell to watch the server log.")

In [ ]:
# Watch the server log — stop this cell (■) once you see "Models ready."
# On Windows, replace 'tail -f' with: !type NUL & python -c "import time; [print(open('/tmp/review.log').read()) or time.sleep(2) for _ in iter(int, 1)]"
!tail -f /tmp/review.log

---
## Step 4: Re-extract entries with bounding boxes

Once you've finished reviewing (or decided the automatic alignment is good enough), re-run entry extraction. This time the extractor reads the `*_aligned.json` files and attaches `#xywh=` bounding box coordinates to every entry it can match.

The resulting `canvas_fragment` URLs in your CSV will point to the exact spot on the original page scan — useful for verification, for building linked data applications, and for the IIIF viewer integration.

In [ ]:
# Re-run entry extraction to pick up the aligned bounding boxes.
# If you used a custom NER prompt from a different volume, add:
#   --ner-prompt output/<first-slug>/ner_prompt.md
!python main.py --extract-entries --model {MODEL} {VOLUME}

In [ ]:
# Check how many entries received bounding box coordinates
import csv
from pathlib import Path

vol_path = Path(PIPELINE_DIR) / VOLUME
csvs = sorted(vol_path.glob(f"entries_{MODEL}*.csv"))

if not csvs:
    print("No entries CSV found — check that extraction completed successfully")
else:
    csv_path = csvs[-1]
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))

    with_bbox    = sum(1 for r in rows if "#xywh=" in r.get("canvas_fragment", ""))
    without_bbox = len(rows) - with_bbox
    pct = with_bbox / len(rows) * 100 if rows else 0

    print(f"CSV: {csv_path.name}")
    print(f"Total entries:        {len(rows)}")
    print(f"With bounding box:    {with_bbox}  ({pct:.0f}%)")
    print(f"Without bounding box: {without_bbox}")

    # Show a sample entry that has a bbox
    sample = next((r for r in rows if "#xywh=" in r.get("canvas_fragment", "")), rows[0] if rows else None)
    if sample:
        print("\nSample entry:")
        for k in ["name", "city", "state", "address", "canvas_fragment"]:
            val = sample.get(k, "")
            if val:
                print(f"  {k}: {val[:90]}{'…' if len(val) > 90 else ''}")

    if without_bbox > len(rows) * 0.2:
        print(f"\nNote: {without_bbox / len(rows):.0%} of entries have no bounding box.")
        print("This is normal for entries on pages that had poor alignment.")
        print("Additional review in Step 3 can recover some of these.")

---
## Troubleshooting

**Surya runs out of memory**  
Lower `--batch-size` (try `4` or `8`) and re-run. Surya skips pages that already have a `_surya.json`, so no work is repeated.

**Surya is very slow**  
It's almost certainly running on CPU. See the GPU requirement note at the top. Even Apple Silicon (M1/M2/M3) is dramatically faster than CPU-only.

**Alignment produces many unmatched lines**  
This usually means a two-column layout where Surya only detected one column, or low scan quality. The review UI prioritises these pages — manually matching a few key lines often helps the algorithm recover the rest.

**Review server won't start or shows an error**  
Check the log file:
```python
!cat /tmp/review.log | tail -30
```
If it says `Address already in use`, kill the old process:
```python
!fuser -k 5000/tcp
```
Then re-run the Start server cell.

**Clicking Done in the review UI shuts down the server**  
This is intentional — Done saves your work and signals completion. To continue reviewing more pages, re-run the Start server cell.

**Lost the localhost URL / need to reconnect**  
The server keeps running in the background. Just open `http://localhost:5000` (or whatever port you set) in your browser again.

**Entries still have no `canvas_fragment` bounding box after re-extraction**  
The extractor matches entry text to aligned lines using fuzzy substring matching. If the NER model's output text diverges significantly from the aligned OCR text (e.g. after heavy paraphrasing), some entries won't find a match. Check that your `*_aligned.json` files were updated after review by looking at their modification timestamps:
```python
import os, glob
for f in sorted(glob.glob(f"{VOLUME}/*_aligned.json"), key=os.path.getmtime)[-5:]:
    print(os.path.getmtime(f), f)
```